# 📊 Model Evaluation & Comparison
## Offline AI Assistant for Smart Farming
### Member 1 - AI/ML Module
---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('Libraries loaded ✅')

## 1. Load Models & Test Data

In [ ]:
# Load crop models and data
crop_rf  = joblib.load('../models/crop_rf_model.pkl')
crop_xgb = joblib.load('../models/crop_xgb_model.pkl')
crop_le  = joblib.load('../models/crop_label_encoder.pkl')

crop_X_test = pd.read_csv('../data/processed/crop_X_test.csv')
crop_y_test = pd.read_csv('../data/processed/crop_y_test.csv').values.ravel()

# Load fertilizer models and data
fert_rf       = joblib.load('../models/fertilizer_rf_model.pkl')
fert_xgb      = joblib.load('../models/fertilizer_xgb_model.pkl')
fert_encoders = joblib.load('../models/fertilizer_encoders.pkl')

fert_X_test = pd.read_csv('../data/processed/fertilizer_X_test.csv')
fert_y_test = pd.read_csv('../data/processed/fertilizer_y_test.csv').values.ravel()

print('All models and test data loaded ✅')

## 2. Crop Model — Full Evaluation

In [ ]:
# Predictions
crop_rf_preds  = crop_rf.predict(crop_X_test)
crop_xgb_preds = crop_xgb.predict(crop_X_test)

# Metrics
def get_metrics(y_true, y_pred, name):
    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2),
        'Recall':    round(recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2),
        'F1 Score':  round(f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2),
    }

crop_results = pd.DataFrame([
    get_metrics(crop_y_test, crop_rf_preds,  'Random Forest'),
    get_metrics(crop_y_test, crop_xgb_preds, 'XGBoost'),
])

print('=== CROP MODEL EVALUATION ===')
crop_results

In [ ]:
# Crop metrics bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, crop_results[crop_results['Model']=='Random Forest'][metrics].values[0],
               width, label='Random Forest', color='steelblue')
bars2 = ax.bar(x + width/2, crop_results[crop_results['Model']=='XGBoost'][metrics].values[0],
               width, label='XGBoost', color='coral')

ax.set_ylabel('Score (%)')
ax.set_title('Crop Model — Random Forest vs XGBoost', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(85, 101)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1.5,
            f'{bar.get_height():.1f}', ha='center', va='top', color='white', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1.5,
            f'{bar.get_height():.1f}', ha='center', va='top', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/crop_model_comparison.png', dpi=150)
plt.show()

## 3. Fertilizer Model — Full Evaluation

In [ ]:
fert_rf_preds  = fert_rf.predict(fert_X_test)
fert_xgb_preds = fert_xgb.predict(fert_X_test)

fert_results = pd.DataFrame([
    get_metrics(fert_y_test, fert_rf_preds,  'Random Forest'),
    get_metrics(fert_y_test, fert_xgb_preds, 'XGBoost'),
])

print('=== FERTILIZER MODEL EVALUATION ===')
fert_results

In [ ]:
# Fertilizer metrics bar chart
fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, fert_results[fert_results['Model']=='Random Forest'][metrics].values[0],
               width, label='Random Forest', color='mediumseagreen')
bars2 = ax.bar(x + width/2, fert_results[fert_results['Model']=='XGBoost'][metrics].values[0],
               width, label='XGBoost', color='mediumpurple')

ax.set_ylabel('Score (%)')
ax.set_title('Fertilizer Model — Random Forest vs XGBoost', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(80, 101)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1.5,
            f'{bar.get_height():.1f}', ha='center', va='top', color='white', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1.5,
            f'{bar.get_height():.1f}', ha='center', va='top', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/fertilizer_model_comparison.png', dpi=150)
plt.show()

## 4. SHAP Feature Importance

In [ ]:
import shap

# SHAP for crop model
explainer_crop  = shap.TreeExplainer(crop_rf)
shap_values_crop = explainer_crop.shap_values(crop_X_test)

plt.figure()
shap.summary_plot(
    shap_values_crop, crop_X_test,
    plot_type='bar',
    class_names=crop_le.classes_,
    show=False
)
plt.title('SHAP Feature Importance — Crop Model')
plt.tight_layout()
plt.savefig('../data/processed/crop_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP crop plot saved')

In [ ]:
# SHAP for fertilizer model
explainer_fert   = shap.TreeExplainer(fert_rf)
shap_values_fert = explainer_fert.shap_values(fert_X_test)

fert_classes = fert_encoders['Fertilizer Name'].classes_

plt.figure()
shap.summary_plot(
    shap_values_fert, fert_X_test,
    plot_type='bar',
    class_names=fert_classes,
    show=False
)
plt.title('SHAP Feature Importance — Fertilizer Model')
plt.tight_layout()
plt.savefig('../data/processed/fertilizer_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP fertilizer plot saved')

## 5. Final Summary

In [ ]:
print('=' * 55)
print('         FINAL MODEL EVALUATION SUMMARY')
print('=' * 55)
print('\n🌾 CROP RECOMMENDATION MODEL')
print(crop_results.to_string(index=False))
print('\n🧪 FERTILIZER RECOMMENDATION MODEL')
print(fert_results.to_string(index=False))

best_crop = crop_results.loc[crop_results['Accuracy'].idxmax(), 'Model']
best_fert = fert_results.loc[fert_results['Accuracy'].idxmax(), 'Model']
print(f'\n🏆 Best Crop Model       : {best_crop}')
print(f'🏆 Best Fertilizer Model : {best_fert}')
print('\n✅ Evaluation Complete!')